In [3]:
# import necessary libraries
import sqlite3
import pandas as pd
import time
from datetime import datetime


## connecting to our OLTP database
conn = sqlite3.connect('nitish_rides.db')
cursor = conn.cursor()    # sql executor

print("OLTP database initialized")

OLTP database initialized


In [4]:
## creating table with constraints
cursor.execute('''
    CREATE TABLE IF NOT EXISTS rides(
        ride_id INTEGER PRIMARY KEY AUTOINCREMENT,
        user_id TEXT NOT NULL,
        driver_id TEXT,
        pickup_location TEXT,
        destination TEXT,
        status TEXT CHECK(status IN ('requested','ongoing','completed','cancelled')),
        Price REAL,
        timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
    )

''')
conn.commit()
print("Table created")

Table created


In [5]:
#CREATING A FUNCTION TO BOOK RIDES
def book_ride(user_id, pickup, dest, price):
    try:
        #start Transaction
        cursor.execute("BEGIN TRANSACTION")
        # Step 1 INSERT RIDE
        cursor.execute('''
            INSERT INTO rides(
                user_id,
                pickup_location,
                destination,
                status,
                Price                 
             )
            VALUES(?,?,?,'requested',?)
        ''',(user_id, pickup, dest, price))

        # simulate payment
        payment_success = True
        if not payment_success:
            raise Exception("payment failed") 
        # save changes
        conn.commit()
        print(f"ride booked successfully for {user_id}")

    except Exception as e:
        # rollback if anything fails
        conn.rollback()
        print(f"Transaction failed due to {e}, No data was saved")



In [6]:
book_ride("nikita_102","mandi","hamirpur",270)

ride booked successfully for nikita_102


In [7]:
##  now lets impliment isolation
def assign_driver(ride_id,driver_id):
    cursor.execute(
        "SELECT status FROM rides WHERE ride_id = ?",(ride_id,)

    )
    status = cursor.fetchone()[0]

    if status == 'requested':
        cursor.execute(
            '''
                UPDATE rides
                SET driver_id = ?, status = 'ongoing'
                WHERE ride_id = ?
        ''',
        (driver_id,ride_id)
        )

        conn.commit()
        print(f"Driver {driver_id} assigned to ride {ride_id}")

    else:
        print("ride already assigned or cancelled")




In [8]:
assign_driver(1,"driver_deepeak")

Driver driver_deepeak assigned to ride 1


In [9]:
def get_features_model(user_id):
    query = f"""
        SELECT AVG(price)
        FROM rides
        WHERE user_id = '{user_id}'
        AND status  = 'completed'
        LIMIT 5
    """
    df = pd.read_sql_query(query,conn)
    return df.iloc[0,0]